In [11]:
import os
from pathlib import Path
import pandas as pd

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
RAW = DATA_DIR / "raw" / "part_b"
OUT = DATA_DIR / "processed" / "part_b_1319"
AGG = OUT / "agg"
AGG.mkdir(parents=True, exist_ok=True)
YEARS = list(range(2013, 2020))

def raw_path(kind, year):
    name = f"Medicare_Physician_Other_Practitioners_by_{kind}_{year}"
    return RAW / name / f"{name}.csv"

NUM = ["Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs", "Avg_Sbmtd_Chrg", "Avg_Mdcr_Pymt_Amt"]
USE = ["Rndrng_NPI", "Rndrng_Prvdr_Type", "Rndrng_Prvdr_Ent_Cd", "HCPCS_Cd", "Place_Of_Srvc"] + NUM
STATS = ["min", "max", "median", "mean", "sum", "std"]

In [12]:
TYPE_MAP = {
    "Allergy/Immunology": "Allergy/ Immunology",
    "Ambulance Service Supplier": "Ambulance Service Provider",
    "Anesthesiologist Assistants": "Anesthesiology Assistant",
    "Audiologist (billing independently)": "Audiologist",
    "CRNA": "Certified Registered Nurse Anesthetist (CRNA)",
    "Cardiac Electrophysiology": "Clinical Cardiac Electrophysiology",
    "Clinical Cardiatric Electrophysiology": "Clinical Cardiac Electrophysiology",
    "Cardiovascular Disease (Cardiology)": "Cardiology",
    "Clinical Psychologist": "Psychologist, Clinical",
    "Colorectal Surgery (formerly proctology)": "Colorectal Surgery (Proctology)",
    "Gynecological/Oncology": "Gynecological Oncology",
    "Hematology/Oncology": "Hematology-Oncology",
    "Independent Diagnostic Testing Facility": "Independent Diagnostic Testing Facility (IDTF)",
    "Mammographic Screening Center": "Mammography Center",
    "Mass Immunization Roster Biller": "Mass Immunizer Roster Biller",
    "Obstetrics/Gynecology": "Obstetrics & Gynecology",
    "Occupational therapist": "Occupational Therapist in Private Practice",
    "Oral Surgery (Dentists only)": "Oral Surgery (Dentist only)",
    "Oral Surgery (dentists only)": "Oral Surgery (Dentist only)",
    "Physical Therapist": "Physical Therapist in Private Practice",
    "Portable X-ray": "Portable X-Ray Supplier",
    "Public Health Welfare Agency": "Public Health or Welfare Agency",
    "Radiation Therapy": "Radiation Therapy Center",
    "Registered Dietician/Nutrition Professional": "Registered Dietitian or Nutrition Professional",
    "Unknown Supplier/Provider": "Unknown Supplier/Provider Specialty",
}
audit = pd.read_csv(OUT / "audit" / "provider_type_by_year.csv", index_col=0)
audit.index = audit.index.str.strip()
assert len(TYPE_MAP) == 25
assert all(k in audit.index for k in TYPE_MAP), [k for k in TYPE_MAP if k not in audit.index]
assert all(audit.loc[v, "2019"] > 0 for v in TYPE_MAP.values())
print("Types after renames:", pd.Series(audit.index).replace(TYPE_MAP).nunique(), "(paper: 102)")

Types after renames: 102 (paper: 102)


In [13]:
def load_clean(year):
    df = pd.read_csv(raw_path("Provider_and_Service", year), usecols=USE, encoding="latin-1",
                     dtype={c: str for c in USE if c not in NUM})
    df["Rndrng_Prvdr_Type"] = df["Rndrng_Prvdr_Type"].str.strip().replace(TYPE_MAP)
    df["year"] = year
    return df

def check(df):
    num_ok = all(pd.api.types.is_numeric_dtype(df[c]) for c in NUM)
    by_npi = df.groupby("Rndrng_NPI")
    return {
        "rows": len(df),
        "nulls": int(df[USE].isna().sum().sum()),
        "numeric_ok": num_ok,
        "npi_not_10_digits": int((~df["Rndrng_NPI"].str.fullmatch(r"\d{10}")).sum()),
        "dup_npi_hcpcs_pos": int(df.duplicated(["Rndrng_NPI", "HCPCS_Cd", "Place_Of_Srvc"]).sum()),
        "npis_multi_type": int((by_npi["Rndrng_Prvdr_Type"].nunique() > 1).sum()),
        "npis_multi_entcd": int((by_npi["Rndrng_Prvdr_Ent_Cd"].nunique() > 1).sum()),
        "negative_values": int((df[NUM] < 0).sum().sum()) if num_ok else None,
    }

In [14]:
df13 = load_clean(2013)
print(check(df13))
print("Provider types in 2013 after renames:", df13["Rndrng_Prvdr_Type"].nunique())

{'rows': 9286633, 'nulls': 0, 'numeric_ok': True, 'npi_not_10_digits': 0, 'dup_npi_hcpcs_pos': 0, 'npis_multi_type': 0, 'npis_multi_entcd': 0, 'negative_values': 0}
Provider types in 2013 after renames: 90


In [15]:
def aggregate(df):
    g = df.groupby(["Rndrng_NPI", "year", "Rndrng_Prvdr_Type"], sort=False)
    agg = g[NUM].agg(STATS)
    agg.columns = [f"{c}_{s}" for c, s in agg.columns]
    std_cols = [c for c in agg.columns if c.endswith("_std")]
    agg[std_cols] = agg[std_cols].fillna(0)
    agg["Rndrng_Prvdr_Ent_Cd"] = g["Rndrng_Prvdr_Ent_Cd"].first()
    return agg.reset_index()

agg13 = aggregate(df13)
print(agg13.shape, "| unique NPIs in 2013:", df13["Rndrng_NPI"].nunique())

(909560, 34) | unique NPIs in 2013: 909560


In [16]:
npi = df13["Rndrng_NPI"].value_counts().index[100]
rows = df13[df13["Rndrng_NPI"] == npi]
manual = {f"{c}_{s}": getattr(rows[c], s)() for c in NUM for s in STATS}
pipe = agg13[agg13["Rndrng_NPI"] == npi].iloc[0]
diff = {k: (v, pipe[k]) for k, v in manual.items() if abs(v - pipe[k]) > 1e-6}
print("NPI", npi, "| rows:", len(rows), "| mismatches:", diff if diff else "none")

NPI 1255319372 | rows: 265 | mismatches: none


In [17]:
del df13, agg13
checks = {}
for y in YEARS:
    df = load_clean(y)
    checks[y] = check(df)
    agg = aggregate(df)
    agg.to_parquet(AGG / f"part_b_agg_{y}.parquet", index=False)
    print(y, "done:", agg.shape)
    del df, agg
checks = pd.DataFrame(checks).T
print(checks)
checks.to_csv(OUT / "audit" / "r02_checks.csv")

2013 done: (909560, 34)
2014 done: (938141, 34)
2015 done: (968333, 34)
2016 done: (1000924, 34)
2017 done: (1032911, 34)
2018 done: (1061278, 34)
2019 done: (1093367, 34)
          rows nulls numeric_ok npi_not_10_digits dup_npi_hcpcs_pos  \
2013   9286633     0       True                 0                 0   
2014   9316209     0       True                 0                 0   
2015   9496848     0       True                 0                 0   
2016   9714896     0       True                 0                 0   
2017   9847443     0       True                 0                 0   
2018   9961865     0       True                 0                 0   
2019  10140228     0       True                 0                 0   

     npis_multi_type npis_multi_entcd negative_values  
2013               0                0               0  
2014               0                0               0  
2015               0                0               0  
2016               0               

In [18]:
agg_all = pd.concat([pd.read_parquet(AGG / f"part_b_agg_{y}.parquet") for y in YEARS], ignore_index=True)
print(agg_all.groupby("year").size())
print("Total rows:", len(agg_all), "| paper (Table 7): 8,669,497")
print("Rows with the same NPI+year (should be 0):", int(agg_all.duplicated(["Rndrng_NPI", "year"]).sum()))
print("Columns:", agg_all.shape[1])
agg_all.to_parquet(OUT / "part_b_agg_2013_2019.parquet", index=False)

year
2013     909560
2014     938141
2015     968333
2016    1000924
2017    1032911
2018    1061278
2019    1093367
dtype: int64
Total rows: 7004514 | paper (Table 7): 8,669,497
Rows with the same NPI+year (should be 0): 0
Columns: 34


In [19]:
pos_counts = {}
for y in YEARS:
    d = pd.read_csv(raw_path("Provider_and_Service", y), usecols=["Rndrng_NPI", "Place_Of_Srvc"],
                    dtype=str, encoding="latin-1")
    print(y, d["Place_Of_Srvc"].value_counts().to_dict())
    pos_counts[y] = len(d.drop_duplicates())
pos_counts = pd.Series(pos_counts)
print(pos_counts)
print("Total NPI+year+POS:", pos_counts.sum(), "| paper: 8,669,497 | NPI+year only: 7,004,514")

2013 {'O': 5556189, 'F': 3730444}
2014 {'O': 5558413, 'F': 3757796}
2015 {'O': 5683111, 'F': 3813737}
2016 {'O': 5972342, 'F': 3742554}
2017 {'O': 6077711, 'F': 3769732}
2018 {'O': 6132242, 'F': 3829623}
2019 {'O': 6252677, 'F': 3887551}
2013    1142101
2014    1167606
2015    1198535
2016    1244353
2017    1275976
2018    1304939
2019    1338209
dtype: int64
Total NPI+year+POS: 8671719 | paper: 8,669,497 | NPI+year only: 7,004,514


In [20]:
KEYS = ["Rndrng_NPI", "year", "Rndrng_Prvdr_Type", "Place_Of_Srvc"]
AGG_POS = OUT / "agg_pos"
AGG_POS.mkdir(parents=True, exist_ok=True)

def aggregate_pos(df):
    g = df.groupby(KEYS, sort=False)
    agg = g[NUM].agg(STATS)
    agg.columns = [f"{c}_{s}" for c, s in agg.columns]
    std_cols = [c for c in agg.columns if c.endswith("_std")]
    agg[std_cols] = agg[std_cols].fillna(0)
    agg["Rndrng_Prvdr_Ent_Cd"] = g["Rndrng_Prvdr_Ent_Cd"].first()
    return agg.reset_index()

In [21]:
df13 = load_clean(2013)
agg13 = aggregate_pos(df13)
print(agg13.shape, "| expected rows: 1,142,101")

both = df13.groupby("Rndrng_NPI")["Place_Of_Srvc"].nunique()
npi = both[both == 2].index[50]
rows = df13[(df13["Rndrng_NPI"] == npi) & (df13["Place_Of_Srvc"] == "F")]
manual = {f"{c}_{s}": getattr(rows[c], s)() for c in NUM for s in STATS}
if len(rows) == 1:
    manual.update({f"{c}_std": 0.0 for c in NUM})
pipe = agg13[(agg13["Rndrng_NPI"] == npi) & (agg13["Place_Of_Srvc"] == "F")].iloc[0]
diff = {k: (v, pipe[k]) for k, v in manual.items() if abs(v - pipe[k]) > 1e-6}
print("NPI", npi, "| F rows:", len(rows), "| mismatches:", diff if diff else "none")

(1142101, 35) | expected rows: 1,142,101
NPI 1003012063 | F rows: 9 | mismatches: none


In [22]:
del df13, agg13
for y in YEARS:
    df = load_clean(y)
    agg = aggregate_pos(df)
    agg.to_parquet(AGG_POS / f"part_b_agg_pos_{y}.parquet", index=False)
    print(y, "done:", agg.shape)
    del df, agg

2013 done: (1142101, 35)
2014 done: (1167606, 35)
2015 done: (1198535, 35)
2016 done: (1244353, 35)
2017 done: (1275976, 35)
2018 done: (1304939, 35)
2019 done: (1338209, 35)


In [23]:
agg_all = pd.concat([pd.read_parquet(AGG_POS / f"part_b_agg_pos_{y}.parquet") for y in YEARS],
                    ignore_index=True)
per_year = agg_all.groupby("year").size()
print(pd.DataFrame({"rows": per_year, "expected": pos_counts, "match": per_year == pos_counts}))
print("Total:", len(agg_all), "| paper: 8,669,497")
print("Duplicate keys (should be 0):", int(agg_all.duplicated(KEYS).sum()))
print("Columns:", agg_all.shape[1])
agg_all.to_parquet(OUT / "part_b_agg_pos_2013_2019.parquet", index=False)

         rows  expected  match
2013  1142101   1142101   True
2014  1167606   1167606   True
2015  1198535   1198535   True
2016  1244353   1244353   True
2017  1275976   1275976   True
2018  1304939   1304939   True
2019  1338209   1338209   True
Total: 8671719 | paper: 8,669,497
Duplicate keys (should be 0): 0
Columns: 35
